# Extract a subset of studies

The competition data is already mounted here. This copies the first `N_STUDIES` of it
into `/kaggle/working` as one archive, so it can be fetched with a **single**
`kaggle kernels output` instead of one API request per slice.

That distinction is the whole point. Asking the file endpoint for a subset costs one
request per `.dcm` — about 16,000 for 80 studies — which exhausts the account's
download quota and blocks *every* download, bulk transfers included. One archive costs
one request.

For the full corpus, do not use this: `kaggle competitions download` without `-f` is
already a single request.

Set `N_STUDIES`, run all, then locally:

```bash
kaggle kernels output mathysgouverneur/rsna-knee-extract -p /tmp/extract
for t in /tmp/extract/train_subset_*.tar; do tar -xf "$t" -C data/raw; done
```

The output is split into parts: `kernels output` does not resume, so a connection
dropped at 3 GB of a 10 GB archive loses all 3 GB. Re-running skips parts already on
disk, so a break costs one part.

In [ ]:
import os
import shutil
import tarfile
import time
from pathlib import Path

import pandas as pd

N_STUDIES = 80
SPLIT = "train_series"

# Kaggle mounts the competition under one of these, depending on the day.
ROOT = next(p for p in (Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                        Path("/kaggle/input/rsna-knee-abnormality-detection"))
            if (p / "test.csv").is_file())
OUT = Path("/kaggle/working")
print("competition root:", ROOT)

## What will be taken

In [ ]:
# Studies in the order the CSV lists them, so the subset is reproducible: the same
# N_STUDIES here and on any other run means the same studies.
series = pd.read_csv(ROOT / f"{SPLIT.replace('_series', '')}_series.csv",
                     dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
studies = sorted(series["StudyInstanceUID"].unique())[:N_STUDIES]

files, size = [], 0
for study in studies:
    for path in (ROOT / SPLIT / study).rglob("*.dcm"):
        files.append(path)
        size += path.stat().st_size

print(f"{len(studies)} studies, {len(files)} slices, {size / 1024 ** 3:.1f} GB")
print(f"{size / 1024 ** 3 / len(studies):.2f} GB per study")

## Does it fit

In [ ]:
# Split into parts of about PART_GB, because `kaggle kernels output` does not resume:
# a connection dropped at 3 GB of a 10 GB archive loses all 3 GB. With parts, a break
# costs one part, and re-running skips the ones already on disk.
PART_GB = 1.5
LIMIT_GB = 18.0

assert size / 1024 ** 3 < LIMIT_GB, (
    f"{size / 1024 ** 3:.1f} GB exceeds the {LIMIT_GB} GB budget. "
    f"Lower N_STUDIES to about {int(N_STUDIES * LIMIT_GB / (size / 1024 ** 3))}.")

# Group whole studies into parts: a study never straddles two archives, so one part
# unpacks into something the pipeline can read even if another never arrives.
parts, current, current_size = [], [], 0
for study in studies:
    study_size = sum(p.stat().st_size for p in (ROOT / SPLIT / study).rglob("*.dcm"))
    if current and current_size + study_size > PART_GB * 1024 ** 3:
        parts.append(current)
        current, current_size = [], 0
    current.append(study)
    current_size += study_size
if current:
    parts.append(current)

print(f"{size / 1024 ** 3:.1f} GB in {len(parts)} part(s) of ~{PART_GB} GB")
for i, part in enumerate(parts):
    print(f"  part {i:02d}: {len(part)} studies")

## Archive it

In [ ]:
# Stored, not compressed: DICOM pixel data is already compressed, so deflating it
# spends minutes to save a few per cent.
t0 = time.time()
written = []

for i, part in enumerate(parts):
    archive = OUT / f"train_subset_{i:02d}.tar"
    with tarfile.open(archive, "w") as tar:
        for study in part:
            tar.add(ROOT / SPLIT / study, arcname=f"{SPLIT}/{study}")
    written.append(archive)
    print(f"  {archive.name}: {len(part)} studies, "
          f"{archive.stat().st_size / 1024 ** 3:.2f} GB, {time.time() - t0:.0f}s",
          flush=True)

total = sum(a.stat().st_size for a in written)
print(f"\n{len(written)} archives, {total / 1024 ** 3:.2f} GB")

## Verify before downloading it

In [ ]:
# Read every archive back and count what it holds. A tar cut short fails here rather
# than on the laptop after a long download.
seen = []
for archive in written:
    with tarfile.open(archive) as tar:
        seen += [m.name for m in tar.getmembers() if m.name.endswith(".dcm")]

per_study = pd.Series([n.split("/")[1] for n in seen]).value_counts()
print(f"{len(seen)} slices across {per_study.size} studies")
print(f"slices per study: min {per_study.min()}, median {int(per_study.median())}, "
      f"max {per_study.max()}")

assert len(seen) == len(files), f"archives hold {len(seen)} of {len(files)} slices"
assert per_study.size == len(studies), "some study is missing entirely"
print("\ncomplete")